Dust cloud at low altitude. Assume less than 5km

In [ ]:
from sympy import *
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import dblquad
# Constants
pi = np.pi
G = 6.67e-11 #m^3/kg/s^2
M = 7.735e22 #kg
R = 1.737e6 #m
Fimp = 11.6e-15 #kg/m^2/s
Y = 1000
S = pi*R**2

umin = 130 #m/s

# Parameters
alpha = 0.91
mmin = 3e-16 #kg
mmax = 1e-8 #kg
mew = 3.4
Mplus = Fimp * Y * S

#theta = pi/4 radians
psi0 = pi/6 #radians

# Setting up variables
m, v, r, theta, phi = symbols('m v r theta phi', real=True, positive=True)

# Defining equation for initial velocity
h = r-R
u = sqrt(v**2 + 2*G*M*(1/R - 1/r))

# Defining equation for minimum initial velocity at certain altitude
#u0 = Piecewise((sqrt(2*G*M*(1/R - 1/r)), h > 4900), (130, True)) # v = 0ms^-1
u0 = sqrt(2*G*M*(1/R - 1/r))

# Defining equation for distribution of initial velocities
fu = ((mew-1)/(u0**(1-mew)))*(u**(-mew))

# Defining equation for ejection cone angle
psi = asin(((r*v) / (R*u)) * sin(theta))

# Defining equation for distribution of ejection angles
fpsi = (sin(psi))/(1-cos(psi0))

# Defining the distribution of the rate of grains produced
mdist = Mplus*(1-alpha)/(mmax**(1-alpha)-mmin**(1-alpha)) * m**(-alpha-1) 
Nplus = integrate(mdist, m)
Nplus = Nplus.subs(m, mmax) - Nplus.subs(m, mmin)

# Phase space density of dust above surface
n = Nplus / (8*pi*R*r) * (fu*fpsi) / (v*u**2*sin(theta)*cos(psi))
n = n*2*pi # Integrating over phi

nFunc = lambdify((v, r, theta), n)


In [ ]:
def calculateThetaMax(r_val):
    umax = 4800
    vmax = sqrt(umax**2+2*G*M*(1/r_val - 1/R))
    ratio = sin(psi0)*R*umax/(r_val*vmax)
    thetamax = asin(ratio)
    return thetamax

def calculateDensity(r_val):
    umax = 4800
    vmax = sqrt(umax**2+2*G*M*(1/r_val - 1/R))
    # Absolute minimum initial velocity is 130ms^-1 so anything below 4900m altitude will have it
    vmin = sqrt(umin**2+2*G*M*(1/r_val - 1/R)) # min u for low altitude is 130
    vmid = sqrt(2400**2+2*G*M*(1/r_val - 1/R)) # max u of not reaching escape velocity

    density1, error1 = dblquad(lambda v, theta: nFunc(v, r_val, theta), vmin, vmid, 0, calculateThetaMax(r_val))
    density2, error2 = dblquad(lambda v, theta: nFunc(v, r_val, theta), vmid, vmax, 0, calculateThetaMax(r_val))
    density = 2*density1 + density2

    return density*2 # theta in opposite direction, symmetry

x1 = np.linspace(1, 1000, 50)
x2 = np.linspace(1, 100, 100)
y1 = [calculateDensity(h+R)/1e2 for h in x1]
y2 = [calculateDensity(h+R)/1e4 for h in x2]

plt.subplot(1,2,1)
plt.plot(x1, y1)
plt.xlabel('Altitude (m)')
plt.ylabel('Density (x$\\mathrm{10}^{2}$ $\\mathrm{m}^{-3}$)')
plt.xlim(0,1000)
plt.ylim(0, 10)
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(x2, y2)
plt.xlabel('Altitude (m)')
plt.ylabel('Density (x$\\mathrm{10}^{4}$ $\\mathrm{m}^{-3}$)')
plt.xlim(0,100)
plt.ylim(0,10)
plt.grid(True)

plt.subplots_adjust(wspace=0.5)

plt.show()